In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 24.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss

from google.colab import files

In [ ]:
uploaded_files = files.upload()

Saving Code_of_Conduct.pdf to Code_of_Conduct.pdf
Saving Time Table_B.Tech I Year I,II Sem (R19) Supplementary Examination August -2026.pdf to Time Table_B.Tech I Year I,II Sem (R19) Supplementary Examination August -2026.pdf
Saving student.pdf to student.pdf
Saving R26_Regulations_B.Tech.pdf to R26_Regulations_B.Tech.pdf


In [ ]:
pdf_files = []

for filename in uploaded_files.keys():
    if filename.lower().endswith(".pdf"):
        pdf_files.append(filename)

print("Uploaded PDF files:")
for pdf in pdf_files:
    print("-", pdf)

print("\nTotal PDFs:", len(pdf_files))

Uploaded PDF files:
- Code_of_Conduct.pdf
- Time Table_B.Tech I Year I,II Sem (R19) Supplementary Examination August -2026.pdf
- student.pdf
- R26_Regulations_B.Tech.pdf

Total PDFs: 4


In [ ]:
documents = []

for pdf_file in pdf_files:

    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text:
            documents.append({
                "text": text,
                "source": pdf_file,
                "page": page_number
            })

print("Total pages extracted:", len(documents))

Total pages extracted: 135


In [ ]:
def preprocess_text(text):

    # Replace multiple spaces with one space
    text = re.sub(r'\s+', ' ', text)

    # Remove unnecessary spaces
    text = text.strip()

    return text


In [ ]:
for doc in documents:
    doc["text"] = preprocess_text(doc["text"])

In [ ]:
print(documents[0]["text"][:1000])

Dear Student... Welcome to Vignan’s Portal of Learning Here, the spirit of the institution instilled among the students has been to seek beyond the mere completion of a course, training for a degree or securing simple employment. All of us, the Staff & Management, wish to see you all grow into :  World-class professionals  Entrepreneurs of great wealth & influence  Leaders in professions of your choice You are all aware that attaining this kind of success or honour is not easy. It is pos- sible only when we are able to impart you training in a holistic way and you receive the same with all earnestness. A holistic approach means much more than a sound educational program. No doubt, a thorough knowledge base with good grades is very essential. But the real value addition takes place through:  Development of Right attitudes - which includes positive thinking, ability to take up challenges & high levels of resilience.  Balanced Behaviour – willingness to give others what you wish to r

In [ ]:
def create_chunks(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = []

for doc in documents:

    text_chunks = create_chunks(doc["text"])

    for chunk in text_chunks:

        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"]
        })

print("Total chunks:", len(chunks))

Total chunks: 155


In [ ]:
chunks[:3]

[{'text': 'Dear Student... Welcome to Vignan’s Portal of Learning Here, the spirit of the institution instilled among the students has been to seek beyond the mere completion of a course, training for a degree or securing simple employment. All of us, the Staff & Management, wish to see you all grow into : \uf06e World-class professionals \uf06e Entrepreneurs of great wealth & influence \uf06e Leaders in professions of your choice You are all aware that attaining this kind of success or honour is not easy. It is pos- sible only when we are able to impart you training in a holistic way and you receive the same with all earnestness. A holistic approach means much more than a sound educational program. No doubt, a thorough knowledge base with good grades is very essential. But the real value addition takes place through: \uf06e Development of Right attitudes - which includes positive thinking, ability to take up challenges & high levels of resilience. \uf06e Balanced Behaviour – willingne

In [ ]:
for i, chunk in enumerate(chunks[:5]):

    print("=" * 80)

    print("Chunk:", i)

    print("Source:", chunk["source"])

    print("Page:", chunk["page"])

    print("Text:")

    print(chunk["text"][:500])

Chunk: 0
Source: Code_of_Conduct.pdf
Page: 3
Text:
Dear Student... Welcome to Vignan’s Portal of Learning Here, the spirit of the institution instilled among the students has been to seek beyond the mere completion of a course, training for a degree or securing simple employment. All of us, the Staff & Management, wish to see you all grow into :  World-class professionals  Entrepreneurs of great wealth & influence  Leaders in professions of your choice You are all aware that attaining this kind of success or honour is not easy. It is pos- sib
Chunk: 1
Source: Code_of_Conduct.pdf
Page: 3
Text:
shape you into well groomed, vibrant & successful professionals, of whom the parents, teachers as well as the nation can be proud of.
Chunk: 2
Source: Code_of_Conduct.pdf
Page: 4
Text:
Code of Conduct for Students 4 What to do? Why? Code of Conduct for Students 5 What to do? Why? 1a. When you are waiting for the bus, the language you speak as well your body language should be decent and dignifi

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

In [ ]:
embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

In [ ]:
embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (155, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)

FAISS index created.
Number of vectors: 155


In [ ]:
def search_documents(query, top_k=5, similarity_threshold=0.45):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    similarities, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for similarity, idx in zip(similarities[0], indices[0]):

        if idx == -1:
            continue

        if similarity < similarity_threshold:
            continue

        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "page": chunks[idx]["page"],
            "similarity": float(similarity)
        })

    return results

In [ ]:
query = "What is the attendance requirement?"

results = search_documents(query)

for i, result in enumerate(results):

    print("=" * 80)

    print("Result:", i + 1)

    print("Source:", result["source"])

    print("Page:", result["page"])

    print("Similarity:", result["similarity"])

    print("Text:")

    print(result["text"][:500])

Result: 1
Source: student.pdf
Page: 8
Similarity: 0.59760582447052
Text:
106 vi) Carrying camera cell phones in to the campus is banned. vii) Every student shall possess the necessary text books and note books. viii) Every student is required to maintain decency without making noise while moving from one classroom to another. Attendance Rules i) Attendance shall be marked daily according to the methods prescribed by the University from time to time. ii) Every student must attend at least 75 per cent of the classes in a semester. A student shall be deemed to have elig
Result: 2
Source: Code_of_Conduct.pdf
Page: 23
Similarity: 0.5688848495483398
Text:
Code of Conduct for Students 22 What to do? Why? Code of Conduct for Students 23 What to do? Why? Great occasions do not make heroes or cowards; they simply unveil them to the eyes of men - Bishop westcott 12e. Every student must attend at least 80 percent of the classes in a semester, otherwise he/she will not be eligible to write the seme

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("LLM loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully.


In [ ]:
def create_prompt(query, results):

    context = ""

    for i, result in enumerate(results):

        context += f"""
SOURCE {i + 1}
Document: {result['source']}
Page: {result['page']}

Content:
{result['text']}

--------------------
"""

    prompt = f"""
You are a University Regulation Assistant.

Your task is to answer the question using ONLY the information
provided in the CONTEXT.

STRICT RULES:
1. Do not use outside knowledge.
2. Do not guess.
3. Do not invent facts, numbers, fees, dates, rules, or policies.
4. If the answer is not clearly supported by the CONTEXT,
   respond exactly:
   Information not found in the uploaded university documents.
5. Give a short answer of 2 to 4 sentences.
6. Do not repeat the answer.
7. Do not mention information that is not present in the context.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    return prompt

In [ ]:
def generate_answer(query, results):

    prompt = create_prompt(query, results)

    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4,
        return_full_text=False
    )

    answer = output[0]["generated_text"].strip()

    return answer

In [ ]:
def rag_answer(query, top_k=5):

    results = search_documents(
        query,
        top_k=top_k,
        similarity_threshold=0.45
    )

    # No sufficiently relevant information found
    if len(results) == 0:

        return {
            "answer": "Information not found in the uploaded university documents.",
            "sources": []
        }

    answer = generate_answer(
        query,
        results
    )

    # Safety check
    not_found_message = (
        "Information not found in the uploaded university documents."
    )

    if not answer or len(answer.strip()) < 5:

        answer = not_found_message

    sources = []

    for result in results:

        source_info = {
            "source": result["source"],
            "page": result["page"]
        }

        if source_info not in sources:
            sources.append(source_info)

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
query = "What is the attendance requirement?"

results = search_documents(query)

for i, result in enumerate(results):

    print("=" * 80)

    print("Result:", i + 1)

    print("Source:", result["source"])

    print("Page:", result["page"])

    print("Similarity:", result["similarity"])

    print("Text:")

    print(result["text"][:500])

Result: 1
Source: student.pdf
Page: 8
Similarity: 0.59760582447052
Text:
106 vi) Carrying camera cell phones in to the campus is banned. vii) Every student shall possess the necessary text books and note books. viii) Every student is required to maintain decency without making noise while moving from one classroom to another. Attendance Rules i) Attendance shall be marked daily according to the methods prescribed by the University from time to time. ii) Every student must attend at least 75 per cent of the classes in a semester. A student shall be deemed to have elig
Result: 2
Source: Code_of_Conduct.pdf
Page: 23
Similarity: 0.5688848495483398
Text:
Code of Conduct for Students 22 What to do? Why? Code of Conduct for Students 23 What to do? Why? Great occasions do not make heroes or cowards; they simply unveil them to the eyes of men - Bishop westcott 12e. Every student must attend at least 80 percent of the classes in a semester, otherwise he/she will not be eligible to write the seme

In [ ]:
question = "What is the attendance requirement?"

response = rag_answer(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(response["answer"])

print("\nSOURCES:")

for source in response["sources"]:
    print(
        f"- {source['source']} | Page {source['page']}"
    )

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'no_repeat_ngram_size', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anywa

QUESTION:
What is the attendance requirement?

ANSWER:
The attendance requirement is stated in the document "Attendance Rules" located in page number 8 of Document Source 1. It states: "Every student shall be entitled to carry a camera phone in his/her room." Additionally, there's a reference to the rule stating: "In order to ensure proper conduct amongst students, every student shall adhere to certain guidelines regarding carrying cameras."

So based on the provided sources, we can conclude that the attendance requirement mentioned in both source documents is consistent across pages 8

SOURCES:
- student.pdf | Page 8
- Code_of_Conduct.pdf | Page 23
- R26_Regulations_B.Tech.pdf | Page 25
- R26_Regulations_B.Tech.pdf | Page 28
- R26_Regulations_B.Tech.pdf | Page 26


In [ ]:
questions = [
    "What is the attendance requirement?",
    "What are the examination rules?",
    "What are the academic regulations?",
    "What is the procedure for examinations?"
]

for question in questions:

    print("\n" + "=" * 80)

    print("QUESTION:")
    print(question)

    response = rag_answer(question)

    print("\nANSWER:")
    print(response["answer"])

    print("\nSOURCES:")

    for source in response["sources"]:
        print(
            f"- {source['source']} | Page {source['page']}"
        )

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
What is the attendance requirement?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
The attendance requirement is stated in the document "Attendance Rules" located in page number 8 of Document Source 1. It states: "Every student shall be entitled to carry a camera phone in his/her room." Additionally, there's a reference to the rule stating: "In order to ensure proper conduct amongst students, every student shall adhere to certain guidelines regarding carrying cameras."

So based on the provided sources, we can conclude that the attendance requirement mentioned in both source documents is consistent across pages 8

SOURCES:
- student.pdf | Page 8
- Code_of_Conduct.pdf | Page 23
- R26_Regulations_B.Tech.pdf | Page 25
- R26_Regulations_B.Tech.pdf | Page 28
- R26_Regulations_B.Tech.pdf | Page 26

QUESTION:
What are the examination rules?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
The Examination Rules provided in this document include several key points related to grading and marking systems used for assessing examinations. Here's how you might summarize these rules succinctly without reproducing specific details:

**Examination Rules Summary**

Examining bodies adhere strictly to the following guidelines regarding grading and marking system for exams:

1. **Formative Assessment**: 
    - Forms of formative assessments included in the rubric are:
      - Classroom participation
      - Participation in group discussions  
      - Completion of

SOURCES:
- R26_Regulations_B.Tech.pdf | Page 28
- R26_Regulations_B.Tech.pdf | Page 32
- R26_Regulations_B.Tech.pdf | Page 49
- R26_Regulations_B.Tech.pdf | Page 26
- R26_Regulations_B.Tech.pdf | Page 30

QUESTION:
What are the academic regulations?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
The academic regulations state that "For the matter(s)
NOT covered herein above" refer to things like carrying cameras etc.,
but also cover issues regarding college entrance exams, dormitory living,
attendance, behavior in libraries/laboratories, smoking/cigarette usage,
etc. They provide detailed procedures and explanations for various sections including how to handle incidents involving shortages of attendance, violations of national credit framework choices based system, discipline actions, and more. 

In summary, it's clear that there are multiple sets

SOURCES:
- R26_Regulations_B.Tech.pdf | Page 49
- student.pdf | Page 8
- R26_Regulations_B.Tech.pdf | Page 4
- student.pdf | Page 7
- R26_Regulations_B.Tech.pdf | Page 50

QUESTION:
What is the procedure for examinations?

ANSWER:
The procedure for examinations includes notifications regarding registration procedures, detail of fee and timetable, clearances of courses with 'I' grade in supplementary exams, threshold values 

In [ ]:
question = "What is the university swimming pool membership fee?"

response = rag_answer(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(response["answer"])

print("\nSOURCES:")

for source in response["sources"]:
    print(
        f"- {source['source']} | Page {source['page']}"
    )

QUESTION:
What is the university swimming pool membership fee?

ANSWER:
Information not found in the uploaded university documents.

SOURCES:


In [ ]:
def display_sources(sources):

    print("\nSources used:")

    for source in sources:

        print(
            f"📄 {source['source']} "
            f"| Page {source['page']}"
        )

In [ ]:
display_sources(response["sources"])


Sources used:


In [ ]:
while True:

    question = input(
        "\nAsk your question (type 'exit' to stop): "
    )

    if question.lower() == "exit":
        break

    response = rag_answer(question)

    print("\nAnswer:")
    print(response["answer"])

    display_sources(
        response["sources"]
    )


Ask your question (type 'exit' to stop): What is the attendance requirement for students?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
Every student must attend 80% of the classes. 

Answered based solely on the provided document content, there is insufficient information available regarding specific attendance requirements for students beyond what's explicitly stated in SOURCE 1 Document: "vi) Carrying a camera phone in to the college premises is forbidden." There isn't enough data in the sources provided to determine exact attendance requirements for every single student across multiple institutions. Therefore, I cannot provide a detailed response including additional details about how many students might need

Sources used:
📄 student.pdf | Page 8
📄 Code_of_Conduct.pdf | Page 23
📄 R26_Regulations_B.Tech.pdf | Page 25
📄 Code_of_Conduct.pdf | Page 22
📄 R26_Regulations_B.Tech.pdf | Page 26

Ask your question (type 'exit' to stop): What is the attendance requirement according to the R26 regulations?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
The attendance requirement according the R2 26 Regulations is 75%. 

To elaborate, based on the provided document, there's specific guidance regarding the attendance requirement stated in Section V):

"vi) The shortageofattendancemaybcondoneupto10percentonthegroundofillhealth,socialobligations,participatingorrepresentinginthosports/culturalevents,placementactivitiesetc." This indicates that "Shortage of attendance up t o 10%"

Sources used:
📄 Code_of_Conduct.pdf | Page 23
📄 R26_Regulations_B.Tech.pdf | Page 25
📄 student.pdf | Page 8
📄 R26_Regulations_B.Tech.pdf | Page 26
📄 R26_Regulations_B.Tech.pdf | Page 50
